In [ ]:
from flask import Flask, render_template, request, redirect, url_for
import sqlite3
import os

app = Flask(__name__)

# Use a fresh DB for the form to avoid schema conflicts with your analytics DB
DATABASE = 'employee_financial_insights.db'  # <-- change from employee_financial_insights.db

def get_db_conn():
    # For simple dev use; for multi-threaded env, consider check_same_thread=False
    return sqlite3.connect(DATABASE)

def init_db():
    # Create DB and table if not exists (simple schema for the form)
    with get_db_conn() as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS employees (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                first_name TEXT NOT NULL,
                last_name TEXT NOT NULL,
                email TEXT,
                department TEXT,
                experience TEXT,
                base_salary_monthly REAL,
                query TEXT NOT NULL
            )
        """)
        conn.commit()

# Initialize database (create table if missing)
init_db()

@app.route('/')
def index():
    # IMPORTANT: no 'templates/' prefix
    return render_template('form.html')

@app.route('/submit', methods=['POST'])
def submit():
    # Flask routes with methods=['POST'] don’t need to check request.method again
    firstname = (request.form.get('firstName') or '').strip()
    lastname = (request.form.get('lastName') or '').strip()
    email = (request.form.get('email') or '').strip()
    department = (request.form.get('department') or '').strip()
    experience = (request.form.get('experience') or '').strip()
    base_salary_monthly_raw = (request.form.get('monthlySalary') or '').strip()
    query = (request.form.get('query') or '').strip()

    # Basic validations
    if not firstname or not lastname or not query:
        return "Error: First name, Last name, and Query are required.", 400

    try:
        base_salary_monthly = float(base_salary_monthly_raw) if base_salary_monthly_raw else None
    except ValueError:
        return "Error: monthlySalary must be a number", 400

    # Save data to SQLite database
    with get_db_conn() as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO employees (first_name, last_name, email, department, experience, base_salary_monthly, query) '
            'VALUES (?, ?, ?, ?, ?, ?, ?)',
            (firstname, lastname, email, department, experience, base_salary_monthly, query)
        )
        conn.commit()

    return redirect(url_for('success'))

@app.route('/success')
def success():
    return 'Form submitted successfully! Data saved to the database.'

if __name__ == '__main__':
    # Use the reloader by default; set use_reloader=False if you don’t want double init in dev
    app.run(debug=True, use_reloader=False)